# Hospital Length-of-Stay Prediction — Exploratory Data Analysis

**Goal**: understand the UCI *Diabetes 130-US Hospitals (1999–2008)* dataset before building a predictive model of inpatient length of stay (LOS).

Data: 101,766 inpatient encounters from 130 US hospitals (real-world EHR, widely used in academic research). Target: `time_in_hospital` (days).

Reference: Strack et al. (2014), *Impact of HbA1c Measurement on Hospital Readmission Rates*.

In [ ]:
import sys
from pathlib import Path
# make the project `src` package importable when the notebook runs
# from anywhere inside the repository
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import matplotlib.pyplot as plt


In [ ]:
from src.config import RAW_DATA_PATH
from src.data_utils import clean_data, load_raw_data

df_raw = load_raw_data()
df = clean_data(df_raw)
print(f'raw   : {df_raw.shape}')
print(f'clean : {df.shape}')
df.head()

## 1. Cleaning decisions
- Dropped `encounter_id` / `patient_nbr` (identifiers) and `weight` (~97% missing), `payer_code` (~40% missing, low signal).
- `age` bins converted to numeric midpoints.
- ICD-9 diagnosis codes collapsed into 19 clinical categories.
- 23 diabetes-medication indicator columns aggregated into `med_count`.
- Prior-visit columns aggregated into `visit_history_total`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

fig, ax = plt.subplots(figsize=(8, 3))
df['time_in_hospital'].hist(bins=range(0, 16), ax=ax, color='#4C72B0', edgecolor='white')
ax.set_title('Distribution of time_in_hospital (days)')
ax.set_xlabel('days')
plt.show()
print(df['time_in_hospital'].describe())

In [ ]:
missing = (df.isna().mean() * 100).sort_values(ascending=False)
missing[missing > 0]

In [ ]:
num_cols = [c for c in df.columns if df[c].dtype in ('int64', 'float64')]
corr = df[num_cols].corrwith(df['time_in_hospital']).sort_values()
fig, ax = plt.subplots(figsize=(7, max(3, len(corr) * 0.35)))
colors = ['#C44E52' if v < 0 else '#55A868' for v in corr.values]
ax.barh(corr.index, corr.values, color=colors)
ax.set_title('Correlation with time_in_hospital')
plt.tight_layout(); plt.show()

In [ ]:
top = df['diag_1_cat'].value_counts().head(12)
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(len(top)), top.values, color='#4C72B0')
ax.set_xticks(range(len(top)))
ax.set_xticklabels(top.index, rotation=30, ha='right')
ax.set_title('Primary diagnosis category (top 12)')
plt.tight_layout(); plt.show()

## Key takeaways (from a full run)
- LOS is heavily right-skewed (median 3 days); log-transform helps.
- Clinical workload (`num_lab_procedures`, `num_medications`, `number_diagnoses`) correlates most strongly with LOS.
- Circulatory / Endocrine / Respiratory diagnoses dominate this cohort.